# 🎙️ Wav2Lip Studio - Khớp Khẩu Hình Video (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Runtime (Thời gian chạy)** -> **Thay đổi loại phần cứng** -> Chọn **T4 GPU**.
> 3. Bấm nút ▶️ ở ô lệnh bên dưới -> Bấm **'Kết nối với Google Drive'** khi được hỏi.
> 4. Lần đầu sẽ lưu model vào Drive, **từ lần thứ 2 trở đi sẽ load tức thì từ Drive trong 3 giây!**

In [ ]:
#@title 🚀 Khởi chạy Wav2Lip WebUI 1-Click (Tự Động Caching Google Drive)
import os
import shutil
from google.colab import drive

# 1. Gắn kết Google Drive thông minh (Nếu đã kết nối thì bỏ qua, tránh lỗi pop-up xác thực)
print("🔗 Đang kiểm tra kết nối Google Drive...")
has_drive = False
if os.path.exists('/content/drive/MyDrive'):
    print("🎉 Google Drive đã được gắn kết từ trước! Bỏ qua xác thực...")
    has_drive = True
else:
    try:
        drive.mount('/content/drive')
        has_drive = True
    except Exception as e:
        print(f"⚠️ Không thể gắn kết Drive tự động ({e}). Sẽ chạy trên bộ nhớ tạm của Colab.")

if has_drive:
    drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/Wav2Lip"
    output_drive_dir = "/content/drive/MyDrive/AI_Colab_Cache/Wav2Lip_Outputs"
    os.makedirs(drive_cache_dir, exist_ok=True)
    os.makedirs(output_drive_dir, exist_ok=True)
else:
    drive_cache_dir = "/content/cache/Wav2Lip"
    output_drive_dir = "/content/cache/Wav2Lip_Outputs"
    os.makedirs(drive_cache_dir, exist_ok=True)
    os.makedirs(output_drive_dir, exist_ok=True)

# 2. Clone mã nguồn Wav2Lip (chỉ lấy commit mới nhất --depth 1 để siêu tốc)
%cd /content
if not os.path.exists("/content/Wav2Lip"):
    print("⚡ Đang nạp mã nguồn Wav2Lip...")
    !git clone --depth 1 https://huggingface.co/camenduru/Wav2Lip /content/Wav2Lip

%cd /content/Wav2Lip
os.makedirs("/content/Wav2Lip/results", exist_ok=True)

# 3. Kiểm tra Caching Model
if os.path.exists(f"{drive_cache_dir}/checkpoints/wav2lip_gan.pth"):
    print("🎉 ĐÃ TÌM THẤY MODEL TRONG GOOGLE DRIVE! Nạp trực tiếp trong 3 giây...")
    !cp -r "{drive_cache_dir}/checkpoints" /content/Wav2Lip/
else:
    print("⏳ Đang lưu bản sao model để lần sau không phải tải lại...")
    os.makedirs(f"{drive_cache_dir}/checkpoints", exist_ok=True)
    !cp -r /content/Wav2Lip/checkpoints "{drive_cache_dir}/"

# 4. Cài đặt thư viện môi trường (Ép buộc nâng cấp Gradio 4+ để fix lỗi Internal Server Error)
print("📦 Đang nâng cấp thư viện lên bản mới nhất...")
!pip install -q -U "gradio>=4.0.0" yt_dlp ffmpeg-python librosa

# 5. Vá lỗi librosa hiện đại trong audio.py
!sed -i "s/librosa.filters.mel(hp.sample_rate, hp.n_fft/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft/g" /content/Wav2Lip/audio.py
!sed -i "s/librosa.core.load/librosa.load/g" /content/Wav2Lip/audio.py

# 6. Khởi động WebUI qua tiến trình Python độc lập (Tránh triệt để xung đột bộ nhớ Jupyter Kernel)
app_code = f'''import os
import shutil
import time
import gradio as gr
from yt_dlp import YoutubeDL

os.makedirs("/content/Wav2Lip/results", exist_ok=True)
output_dir = "{output_drive_dir}"
os.makedirs(output_dir, exist_ok=True)

def sync_lips(video_file, audio_file, youtube_url, pads_top, pads_bottom, pads_left, pads_right):
    target_video = "/content/input_video.mp4"
    if video_file is not None:
        shutil.copy(video_file, target_video)
    elif youtube_url and len(youtube_url.strip()) > 5:
        ydl_opts = {{"overwrites": True, "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4", "outtmpl": target_video}}
        with YoutubeDL(ydl_opts) as ydl:
            ydl.download(youtube_url)
    else:
        return None, "❌ Vui lòng tải lên video hoặc dán link YouTube!"
    
    if audio_file is None:
        return None, "❌ Vui lòng tải lên file âm thanh tiếng Việt!"
    
    output_path = "/content/Wav2Lip/results/result_voice.mp4"
    cmd = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face '{{target_video}}' --audio '{{audio_file}}' --pads {{pads_top}} {{pads_bottom}} {{pads_left}} {{pads_right}} --outfile '{{output_path}}'"
    os.system(cmd)
    
    if os.path.exists(output_path):
        ts_name = f"lipsync_{{int(time.time())}}.mp4"
        shutil.copy(output_path, f"{{output_dir}}/{{ts_name}}")
        return output_path, f"🎉 Khớp khẩu hình thành công! Đã tự động lưu ({ts_name})."
    else:
        return None, "❌ Có lỗi trong quá trình xử lý. Hãy kiểm tra lại định dạng video/âm thanh!"

with gr.Blocks(title="Wav2Lip Cloud Studio") as demo:
    gr.Markdown("## 🎬 Wav2Lip AI - Khớp Khẩu Hình Siêu Tốc (Tesla T4 Cloud GPU)")
    gr.Markdown("Tải lên video gốc của bạn và file âm thanh lồng tiếng tiếng Việt để AI tự động đồng bộ cử động môi theo nhịp nói.")
    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="1. Tải lên Video gốc (MP4)")
            yt_input = gr.Textbox(label="Hoặc dán Link YouTube nếu không tải video lên", placeholder="https://youtu.be/...")
            audio_input = gr.Audio(label="2. Tải lên File Âm thanh Tiếng Việt (MP3 / WAV)", type="filepath")
            with gr.Accordion("Tùy chỉnh đệm viền môi (Pads) - Mặc định chuẩn", open=False):
                p_top = gr.Slider(0, 20, value=0, step=1, label="Pads Top")
                p_bottom = gr.Slider(0, 20, value=10, step=1, label="Pads Bottom (Đệm cằm)")
                p_left = gr.Slider(0, 20, value=0, step=1, label="Pads Left")
                p_right = gr.Slider(0, 20, value=0, step=1, label="Pads Right")
            btn_run = gr.Button("🚀 Bắt Đầu Khớp Khẩu Hình (Generate Lip-Sync)", variant="primary")
        with gr.Column():
            video_output = gr.Video(label="Video Kết Quả (Đã Khớp Môi)")
            status_msg = gr.Textbox(label="Trạng thái xử lý", interactive=False)
    btn_run.click(sync_lips, inputs=[video_input, audio_input, yt_input, p_top, p_bottom, p_left, p_right], outputs=[video_output, status_msg])

if __name__ == '__main__':
    demo.queue().launch(share=True)
'''

with open('/content/Wav2Lip/app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("🚀 Đang khởi động Wav2Lip WebUI...")
!python /content/Wav2Lip/app.py
